In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
import requests

# Configs

In [ ]:
HTTP_PROXY = 'http://sia-lb.telekom.de:8080'
HTTPS_PROXY = 'http://sia-lb.telekom.de:8080'

os.environ['http_proxy'] = HTTP_PROXY
os.environ['https_proxy'] = HTTPS_PROXY

activities_url = 'https://www.strava.com/api/v3/athlete/activities'
auth_url = 'https://www.strava.com/oauth/token'


load_dotenv()
CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')
REFRESH_TOKEN = os.getenv('REFRESH_TOKEN')
GRANT_TYPE = os.getenv('GRANT_TYPE')


root_path = Path.cwd().parent
docs_path = root_path / 'docs' / 'strava'
print(docs_path)

# Get Access Token

In [ ]:
payload = {
    'client_id': CLIENT_ID,
    'client_secret': CLIENT_SECRET,
    'refresh_token': REFRESH_TOKEN,
    'grant_type': GRANT_TYPE,
    'f': 'json',
}

print('Requesting Token...\n')
res = requests.post(auth_url, data=payload, verify=False)
access_token = res.json()['access_token']
print(f'Access Token = {access_token} \n')

header = {'Authorization': 'Bearer ' + access_token}

# Get data from Strava API

In [ ]:
request_page_num = 1
all_activities = []

while True:
    param = {'per_page': 200, 'page': request_page_num}
    my_dataset = requests.get(activities_url, headers=header, params=param).json()

    if len(my_dataset) == 0:
        break

    if all_activities:
        all_activities.extend(my_dataset)

    else:
        all_activities = my_dataset

    request_page_num += 1

print('The length of all activies is: ', len(all_activities))

In [ ]:
df_all_activities = pd.json_normalize(all_activities)
df_all_activities

In [ ]:
df_temp = df_all_activities.copy()
df_temp = df_temp.map(lambda x: str(x) if isinstance(x, (list, dict)) else x)

doc = pd.DataFrame({
    'Column': df_all_activities.columns,
    'Dtype': df_all_activities.dtypes.values,
    'Unique Count': df_temp.nunique().values,
    'Null Count': df_all_activities.isnull().sum().values,
    'Missing %': (df_all_activities.isnull().mean() * 100).round(2).values,
    'Example Value': df_all_activities.iloc[0].values,
    'Description': 'TODO: Add description for each column',
})

In [ ]:
doc

In [ ]:
doc.to_csv(docs_path / 'doc_strava_data.csv', index=False)

# Read docs from CSV

In [46]:
df_docs = pd.read_csv(docs_path / 'doc_strava_data.csv')

In [49]:
df_docs.head()

,Column,Dtype,Unique Count,Null Count,Missing %,Example Value,Description
0,resource_state,int64,1,0,0.0,2,TODO: Add description for each column
1,name,object,67,0,0.0,Hill Intervals,TODO: Add description for each column
2,distance,float64,191,0,0.0,7502.8,TODO: Add description for each column
3,moving_time,int64,188,0,0.0,2402,TODO: Add description for each column
4,elapsed_time,int64,189,0,0.0,2405,TODO: Add description for each column


# Add descriptions to the DataFrame

In [ ]:
# Add a description for each column
descriptions = {
    'id': 'Unique identifier for the activity',
    'resource_state': 'Resource state of the activity',
    'external_id': 'External ID of the activity',
    'upload_id': 'Upload ID of the activity',
    'athlete_id': 'ID of the athlete who performed the activity',
    'name': 'Name of the activity',
    'distance': 'Distance covered in meters',
    'moving_time': 'Moving time in seconds',
    'elapsed_time': 'Elapsed time in seconds',
    'total_elevation_gain': 'Total elevation gain in meters',
    'type': 'Type of activity (e.g., Run, Ride)',
    'start_date': 'Start date and time of the activity',
    'start_date_local': 'Local start date and time of the activity',
    # Add more descriptions as needed
}

In [ ]:
df_docs.to_csv(docs_path / 'doc_strava_data.csv', index=False)